# Document Parsing Protocol(DPP) 정리: 표·양식·차트를 마크다운으로 바꿀 때의 140개 규칙과, 스킬 파일로 Claude·ChatGPT에 붙여 문서 파싱 품질을 맞추는 법

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `ANTHROPIC_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['ANTHROPIC_API_KEY']:
    os.environ[name] = userdata.get(name)

### 저장소 클론과 스킬 구성·규칙 수 확인

In [ ]:
%%bash
git clone --depth 1 https://github.com/run-llama/document_parsing_protocol
cd document_parsing_protocol
ls -l resources/dpp-parse resources/dpp-parse/references
head -4 resources/dpp-parse/SKILL.md | cut -c1-200
grep -o '^\*\*§[0-9]*' resources/dpp-parse/SKILL.md | sort | uniq -c | sort -k2 -V
grep -c '^\*\*§' resources/dpp-parse/SKILL.md

### poppler 설치와 공개 PDF 받기

In [ ]:
%%bash
apt-get install -y poppler-utils >/dev/null 2>&1
mkdir -p /content/pages && cd /content
curl -sL -o fw9.pdf https://www.irs.gov/pub/irs-pdf/fw9.pdf
curl -sL -o parsebench.pdf https://arxiv.org/pdf/2604.08538
sha256sum fw9.pdf parsebench.pdf

### PDF 한 페이지씩 PNG 로 변환

In [ ]:
%%bash
cd /content
pdftoppm -png -r 150 -f 2 -l 2 parsebench.pdf pages/arxiv
pdftoppm -png -r 150 -f 1 -l 1 fw9.pdf pages/w9
ls -l pages

### 비교 스크립트 작성 (ANTHROPIC_API_KEY 필요)

In [ ]:
%%bash
pip install -q anthropic
cat > /content/run.py <<'E'
import anthropic,base64,time
c=anthropic.Anthropic()
skill=open('/content/document_parsing_protocol/public/downloads/DPP-0.1-SKILL.md').read()
PROMPT='이 페이지를 마크다운으로 변환해 줘.'
for name,img in [('arxiv','/content/pages/arxiv-02.png'),('w9','/content/pages/w9-1.png')]:
    b=base64.b64encode(open(img,'rb').read()).decode()
    for mode in ['plain','dpp']:
        kw={'system':skill} if mode=='dpp' else {}
        with c.messages.stream(model='claude-sonnet-5-5',max_tokens=16000,messages=[{'role':'user','content':[{'type':'image','source':{'type':'base64','media_type':'image/png','data':b}},{'type':'text','text':PROMPT}]}],**kw) as s:
            r=s.get_final_message()
        txt=''.join(x.text for x in r.content if x.type=='text')
        open(f'/content/out_{name}_{mode}.md','w').write(txt)
        print(name,mode,r.usage.input_tokens,r.usage.output_tokens,'html_table',txt.count('<table'),'rowspan',txt.count('rowspan'),'[ ]',txt.count('[ ]'),'[x]',txt.count('[x]'))
E

### 규칙 없이/있을 때 실행 후 표 확인

In [ ]:
%%bash
cd /content && python3 run.py
head -12 out_arxiv_plain.md | cut -c1-150
echo ======
head -8 out_arxiv_dpp.md | cut -c1-200

### W-9 체크박스 비교

In [ ]:
%%bash
cd /content
grep -n '☐\|\[ \]\|\[x\]\|blank' out_w9_plain.md | head -12
echo ======
grep -n '\[ \]\|\[x\]\|blank' out_w9_dpp.md | head -12